## Step 9c - OCR + captions with vLLM + Qwen2.5-VL-3B (fast drop-in for step 9)

Same model, same prompt, same per-video JSON as step 9 - but served through
**vLLM continuous batching** instead of one-by-one `model.generate`: ~5-10x
faster (~0.3-0.6 s/frame/GPU on T4), plus the image budget is capped via
`max_pixels` so vision tokens stay small. Videos are sharded across both T4s
by subprocess (one vLLM engine per GPU - KHÔNG dùng tensor_parallel trên T4,
PCIe P2P của Kaggle hay treo).

~244k frames ≈ 2 sessions on T4 x2; with `FRAME_STRIDE = 2` one session.
Resumable per video AND per frame - re-run to continue.

**Outputs** (identical to step 9, drop-in):

```
ocr_caption/<video_name>.json   # { "<frame_id>": {"ocr": "...", "caption": "..."} }
ocr_data.json                   # merged -> backend data/ocr/  (THAY file cũ)
caption_data.json               # merged -> backend data/captions/
```

Set **Accelerator = GPU T4 x2**.

In [ ]:
# vLLM kéo torch riêng của nó - đừng cài thêm torch. Ghim bản đã hỗ trợ
# Qwen2.5-VL và còn chạy được sm75 (T4). Nếu Kaggle image mới lỗi dependency,
# thử nới xuống vllm==0.8.5.
!pip install -q "vllm==0.9.2" qwen-vl-utils

In [ ]:
import os

# Step 3 keyframes (nested: Keyframes/<batch>/<video_name>/<idx>.webp)
KEYFRAMES_DIR = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR    = './ocr_caption'   # one <video_name>.json per video (= step 9)
FRAME_STRIDE  = 1                 # 1 = mọi keyframe; 2 = nửa chi phí
CHUNK         = 192               # số request đẩy vào engine mỗi lượt (vLLM tự batch)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
%%writefile worker_vllm.py
"""OCR+caption worker: một vLLM engine / một GPU, shard video theo chẵn lẻ.

Chạy bằng: CUDA_VISIBLE_DEVICES=<gpu> python worker_vllm.py \
    <shard> <n_shards> <keyframes_dir> <output_dir> <frame_stride> <chunk>
"""
import json
import re
import sys
import time
from pathlib import Path

from PIL import Image
from vllm import LLM, SamplingParams

SHARD, N_SHARDS = int(sys.argv[1]), int(sys.argv[2])
KEYFRAMES_DIR, OUTPUT_DIR = sys.argv[3], sys.argv[4]
FRAME_STRIDE, CHUNK = int(sys.argv[5]), int(sys.argv[6])

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp')

# Prompt GIỐNG HỆT step 9 - output hoán đổi được cho nhau.
PROMPT = (
    "You are analyzing one keyframe from a Vietnamese TV broadcast.\n"
    "Return ONLY a JSON object with exactly these keys:\n"
    '{"ocr": "<ALL text visible in the image, verbatim, keep Vietnamese diacritics; '
    '\"\" if none>",\n'
    ' "caption": "<1-2 English sentences describing the visual scene: subjects, '
    'actions, setting. Do not mention the channel logo or timestamps.>"}\n'
    "No markdown, no explanations, JSON only."
)
# Template chat của Qwen2.5-VL với đúng 1 ảnh (multi_modal_data cấp ảnh thật).
CHAT = ('<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>'
        + PROMPT + '<|im_end|>\n<|im_start|>assistant\n')

llm = LLM(
    model=MODEL_ID,
    dtype='half',                 # T4 không có bf16
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,           # sm75: tránh backend compile/FA không hỗ trợ
    limit_mm_per_prompt={'image': 1},
    # Chặn trần chi phí vision: ~640 patch 28x28 ≈ ảnh 900x560. Frame broadcast
    # 1280x720 bị thu nhỏ vừa đủ đọc chữ lớn + tả cảnh, token giảm ~2-3x.
    mm_processor_kwargs={'min_pixels': 256 * 28 * 28, 'max_pixels': 640 * 28 * 28},
)
SAMPLING = SamplingParams(temperature=0.0, max_tokens=220)
print(f'[shard {SHARD}] engine ready', flush=True)


def _parse_json(text):
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not m:
        return {'ocr': '', 'caption': ''}
    try:
        d = json.loads(m.group(0))
    except ValueError:
        return {'ocr': '', 'caption': ''}
    return {'ocr': str(d.get('ocr', '')).strip(),
            'caption': str(d.get('caption', '')).strip()}


video_dirs = sorted(d for d in Path(KEYFRAMES_DIR).glob('*/*') if d.is_dir())
video_dirs = [d for i, d in enumerate(video_dirs) if i % N_SHARDS == SHARD]
print(f'[shard {SHARD}] {len(video_dirs)} videos | stride={FRAME_STRIDE}', flush=True)

t0, n_done = time.time(), 0
for vi, video_dir in enumerate(video_dirs, 1):
    video_name = video_dir.name
    out_path = Path(OUTPUT_DIR) / f'{video_name}.json'
    done = json.loads(out_path.read_text(encoding='utf-8')) if out_path.exists() else {}
    frames = sorted(p for p in video_dir.iterdir()
                    if p.suffix.lower() in IMAGE_EXTS)[::FRAME_STRIDE]
    todo = [(f'{video_name}_{p.stem}', p) for p in frames
            if f'{video_name}_{p.stem}' not in done]
    for b in range(0, len(todo), CHUNK):
        batch = todo[b:b + CHUNK]
        reqs = []
        for _, p in batch:
            img = Image.open(p).convert('RGB')
            reqs.append({'prompt': CHAT, 'multi_modal_data': {'image': img}})
        try:
            outs = llm.generate(reqs, SAMPLING, use_tqdm=False)
            for (frame_id, _), o in zip(batch, outs):
                done[frame_id] = _parse_json(o.outputs[0].text)
        except Exception as e:            # một chunk hỏng không dừng cả shard
            print(f'[WARN shard {SHARD}] {video_name}: {type(e).__name__}: {e}',
                  flush=True)
            for frame_id, _ in batch:
                done.setdefault(frame_id, {'ocr': '', 'caption': ''})
        out_path.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')
    n_done += len(todo)
    rate = n_done / max(time.time() - t0, 1e-9)
    print(f'[shard {SHARD}] {vi}/{len(video_dirs)} {video_name}: '
          f'+{len(todo)} frames | {rate:.1f} f/s', flush=True)

print(f'[shard {SHARD}] DONE {n_done} frames in {(time.time() - t0) / 3600:.2f} h',
      flush=True)

In [ ]:
# Mỗi GPU một engine, chia video chẵn/lẻ. Resume: chạy lại cell để tiếp tục.
import os
import subprocess
import sys
import time

import torch

N_GPU = max(torch.cuda.device_count(), 1)
procs = []
for shard in range(N_GPU):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard))
    log = open(f'shard{shard}.log', 'a')
    procs.append(subprocess.Popen(
        [sys.executable, 'worker_vllm.py', str(shard), str(N_GPU),
         KEYFRAMES_DIR, OUTPUT_DIR, str(FRAME_STRIDE), str(CHUNK)],
        env=env, stdout=log, stderr=subprocess.STDOUT))
print(f'{N_GPU} workers started')

while any(p.poll() is None for p in procs):
    time.sleep(120)
    for shard in range(N_GPU):
        lines = open(f'shard{shard}.log').read().strip().splitlines()
        print(lines[-1] if lines else f'[shard {shard}] no output yet')
for p in procs:
    print('exit code:', p.wait())

In [ ]:
# Merge: ocr_data.json (format step 6, thay thẳng file cũ) + caption_data.json
import json
from pathlib import Path

ocr_merged, cap_merged = {}, {}
for f in sorted(Path(OUTPUT_DIR).glob('*.json')):
    for frame_id, d in json.loads(f.read_text(encoding='utf-8')).items():
        if d.get('ocr'):
            ocr_merged[frame_id] = d['ocr']
        if d.get('caption'):
            cap_merged[frame_id] = d['caption']

with open('ocr_data.json', 'w', encoding='utf-8') as fh:
    json.dump(ocr_merged, fh, ensure_ascii=False)
with open('caption_data.json', 'w', encoding='utf-8') as fh:
    json.dump(cap_merged, fh, ensure_ascii=False)
print(f'ocr_data.json: {len(ocr_merged)} | caption_data.json: {len(cap_merged)}')

## Use with the backend

```text
ocr_data.json     -> backend/data/ocr/ocr_data.json        (THAY file EasyOCR cũ)
caption_data.json -> backend/data/captions/caption_data.json
```

Rồi build lại các index text + graph (OCR đổi nên scene text trong graph cũng đổi):

```bash
cd data_processing
python 07_build_text_indexes.py    # ocr_text + asr_text + caption_text index
python 08_build_graph.py           # scene.ocr lấy từ OCR mới
```

Có thể chạy TRƯỚC step 9b (Florence): hai step ghi cùng `caption_data.json`,
bản nào chạy sau thì thay bản trước - Qwen giàu chi tiết hơn, Florence nhanh hơn.